# Visão Geral do Notebook
Este notebook realiza a extração, unificação e exportação das variáveis alvo do Censo 2022.

- **Célula 1:** Configuração do ambiente e definição das variáveis.
- **Célula 2:** Leitura dos dados essenciais dos arquivos CSV.
- **Célula 3:** Junção dos DataFrames para criar a base unificada.
- **Célula 4:** Exportação da base final para SQLite e Excel.

In [ ]:
# Célula 1: Configuração do ambiente e definição das variáveis
import pandas as pd  # Importa a biblioteca pandas para manipulação de dados
import sqlite3       # Importa a biblioteca sqlite3 para trabalhar com bancos de dados SQLite
import os            # Importa a biblioteca os para operações com o sistema de arquivos

print("Célula 1: Configurando ambiente e variáveis alvo...")  # Mensagem de início da configuração

caminho_dados = '../dados/'  # Define o caminho para a pasta de dados
caminho_bd = '../banco_de_dados/'  # Define o caminho para a pasta do banco de dados
os.makedirs(caminho_bd, exist_ok=True)  # Cria a pasta do banco de dados se não existir

# Listas estritas com as variáveis que mapeamos no Relatório Metodológico
colunas_basico = ['CD_SETOR', 'NM_MUN', 'NM_BAIRRO', 'SITUACAO', 'v0001', 'v0005']  # Variáveis básicas
colunas_dom1 = ['CD_setor', 'V00001']  # Variáveis de características domiciliares 1
colunas_dom2 = ['setor', 'V00112', 'V00113', 'V00114', 'V00115', 'V00116', 'V00117', 'V00118',
                'V00312', 'V00313', 'V00314', 'V00315', 'V00316', 
                'V00398', 'V00399', 'V00400', 'V00401', 'V00402']  # Variáveis de características domiciliares 2
colunas_alfab = ['CD_setor', 'V00900', 'V00901']  # Variáveis de alfabetização
colunas_raca = ['CD_SETOR', 'V01318', 'V01320', 'V01321']  # Variáveis de cor ou raça
colunas_renda = ['CD_SETOR', 'V06001', 'V06004']  # Variáveis de renda

print("Variáveis alvo definidas com sucesso!")  # Mensagem de sucesso na definição das variáveis

Célula 1: Configurando ambiente e variáveis alvo...
Variáveis alvo definidas com sucesso!


## Célula 2: Leitura dos Dados
Carrega apenas as colunas essenciais dos arquivos CSV, protegendo a memória RAM e otimizando o processamento.

In [ ]:
# Célula 2: Leitura cirúrgica dos dados
print("Célula 2: Lendo os arquivos CSV originais (apenas colunas alvo)...")  # Mensagem de início da leitura

# A chave dtype garante que o CD_SETOR seja lido como texto para não perder os zeros à esquerda
df_basico = pd.read_csv(
    caminho_dados + 'Agregados_por_setores_basico_BR_20250417.csv',  # Caminho do arquivo CSV de dados básicos
    sep=';',  # Delimitador do arquivo CSV
    dtype={'CD_SETOR': str},  # Define o tipo da coluna CD_SETOR como string
    usecols=colunas_basico,  # Carrega apenas as colunas especificadas
    encoding='latin1'  # Define a codificação do arquivo
    )

df_dom1 = pd.read_csv(caminho_dados + 'Agregados_por_setores_caracteristicas_domicilio1_BR.csv', sep=';', dtype={'CD_setor': str}, usecols=colunas_dom1)  # Lê dados de características domiciliares 1
df_dom1 = df_dom1.rename(columns={'CD_setor': 'CD_SETOR'})  # Renomeia a coluna para padronizar

df_dom2 = pd.read_csv(caminho_dados + 'Agregados_por_setores_caracteristicas_domicilio2_BR_20250417.csv', sep=';', dtype={'setor': str}, usecols=colunas_dom2)  # Lê dados de características domiciliares 2
df_dom2 = df_dom2.rename(columns={'setor': 'CD_SETOR'})  # Renomeia a coluna para padronizar

df_alfab = pd.read_csv(caminho_dados + 'Agregados_por_setores_alfabetizacao_BR.csv', sep=';', dtype={'CD_setor': str}, usecols=colunas_alfab)  # Lê dados de alfabetização
df_alfab = df_alfab.rename(columns={'CD_setor': 'CD_SETOR'})  # Renomeia a coluna para padronizar

df_raca = pd.read_csv(caminho_dados + 'Agregados_por_setores_cor_ou_raca_BR.csv', sep=';', dtype={'CD_SETOR': str}, usecols=colunas_raca)  # Lê dados de cor ou raça

df_renda = pd.read_csv(caminho_dados + 'Agregados_por_setores_renda_responsavel_BR.csv', sep=';', dtype={'CD_SETOR': str}, usecols=colunas_renda)  # Lê dados de renda

print("Leitura cirúrgica concluída! Memória RAM protegida.")  # Mensagem de conclusão da leitura

Célula 2: Lendo os arquivos CSV originais (apenas colunas alvo)...


C:\Users\Pedro\AppData\Local\Temp\ipykernel_9908\2259337811.py:12: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dom1 = pd.read_csv(caminho_dados + 'Agregados_por_setores_caracteristicas_domicilio1_BR.csv', sep=';', dtype={'CD_setor': str}, usecols=colunas_dom1)
C:\Users\Pedro\AppData\Local\Temp\ipykernel_9908\2259337811.py:15: DtypeWarning: Columns (227) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dom2 = pd.read_csv(caminho_dados + 'Agregados_por_setores_caracteristicas_domicilio2_BR_20250417.csv', sep=';', dtype={'setor': str}, usecols=colunas_dom2)
C:\Users\Pedro\AppData\Local\Temp\ipykernel_9908\2259337811.py:18: DtypeWarning: Columns (257) have mixed types. Specify dtype option on import or set low_memory=False.
  df_alfab = pd.read_csv(caminho_dados + 'Agregados_por_setores_alfabetizacao_BR.csv', sep=';', dtype={'CD_setor': str}, usecols=colunas_alfab)
C:\Users\Pedro\AppData\Local\Temp\i

Leitura cirúrgica concluída! Memória RAM protegida.


## Célula 3: Junção dos DataFrames
Realiza o join dos DataFrames para criar uma base unificada com todas as variáveis essenciais.

In [ ]:
# Célula 3: Join unificado
print("Célula 3: Realizando o Join para criar a base unificada...")  # Mensagem de início do processo de união

# Começamos com a malha básica e vamos colando os outros dados usando o código do setor
df_ivs = df_basico.merge(df_dom1, on='CD_SETOR', how='left')  # Junta dados de características domiciliares 1
df_ivs = df_ivs.merge(df_dom2, on='CD_SETOR', how='left')      # Junta dados de características domiciliares 2
df_ivs = df_ivs.merge(df_alfab, on='CD_SETOR', how='left')     # Junta dados de alfabetização
df_ivs = df_ivs.merge(df_raca, on='CD_SETOR', how='left')       # Junta dados de cor ou raça
df_ivs = df_ivs.merge(df_renda, on='CD_SETOR', how='left')      # Junta dados de renda

print(f"Join concluído! O Banco IVS final tem {df_ivs.shape[0]} setores censitários e {df_ivs.shape[1]} variáveis essenciais.")  # Mensagem de sucesso com quantidade de setores e variáveis

Célula 3: Realizando o Join para criar a base unificada...
Join concluído! O Banco IVS final tem 468099 setores censitários e 31 variáveis essenciais.


## Célula 4: Exportação dos Resultados
Exporta a base final para um banco de dados SQLite e para um arquivo Excel, garantindo acesso fácil e rápido aos dados analíticos.

In [ ]:
# Célula 4: Exportação para SQLite e Excel
print("Célula 4: Exportando a base final...")  # Mensagem de início da exportação

# 1. Exportando para SQLite
caminho_sqlite = caminho_bd + 'Banco_IVS_Essencial.db'  # Define o caminho do arquivo SQLite
conexao = sqlite3.connect(caminho_sqlite)  # Abre conexão com o banco de dados SQLite
df_ivs.to_sql('base_ivs_bruta', conexao, if_exists='replace', index=False)  # Exporta o DataFrame para o banco de dados
conexao.close()  # Fecha a conexão com o banco de dados
print(f"-> Base guardada no SQLite: {caminho_sqlite}")  # Mensagem de sucesso da exportação para SQLite

# 2. Exportando para Excel
caminho_excel = caminho_bd + 'Base_IVS_Essencial.xlsx'  # Define o caminho do arquivo Excel
print(f"-> Gerando o ficheiro Excel (Isto pode demorar vários minutos devido às 450 mil linhas, aguarde...)")  # Aviso sobre tempo de processamento

# O motor xlsxwriter é mais rápido e confiável para bases grandes
with pd.ExcelWriter(caminho_excel, engine='xlsxwriter') as writer:  # Abre o escritor Excel com o motor xlsxwriter
    df_ivs.to_excel(writer, sheet_name='Base_IVS', index=False)  # Exporta o DataFrame para Excel

print("\nProcesso finalizado com sucesso! O seu banco de dados analítico está pronto.")  # Mensagem de conclusão do processo

Célula 4: Exportando a base final...
-> Base guardada no SQLite: ../banco_de_dados/Banco_IVS_Essencial.db
-> Gerando o ficheiro Excel (Isto pode demorar vários minutos devido às 450 mil linhas, aguarde...)

Processo finalizado com sucesso! O seu banco de dados analítico está pronto.
